# Model: Attention CNN (CBAM)\n**Dataset:** MNIST\n**Task:** Multiclass Image Classification (10 classes)

**Học viện Công nghệ Bưu chính Viễn thông (PTIT) — Khoa CNTT 1**
- **Sinh viên:** Nguyễn Nam Hải (B23DCCN277 - D23CTPM01 - CT01)
- **GVHD:** PGS.TS. Trần Đình Quế
- **GitHub Repository:** [https://github.com/HandQ2212/intel-sys-assignment-05](https://github.com/HandQ2212/intel-sys-assignment-05)
- **Kaggle Dataset:** [400k Augmented MNIST Extended Handwritten Digits Dataset](https://www.kaggle.com/datasets/alexandrelemercier/400k-augmented-mnist-extended-handwritten-digits)


# Lý thuyết: Attention CNN (CBAM)

Cơ chế Attention giúp mạng tập trung vào các đặc trưng quan trọng nhất. CBAM kết hợp hai loại attention:

**1. Channel Attention (WHAT):**
Tập trung vào "kênh nào" là quan trọng:
$$ M_c(F) = \sigma(MLP(AvgPool(F)) + MLP(MaxPool(F))) $$

**2. Spatial Attention (WHERE):**
Tập trung vào "vị trí nào" mang thông tin quan trọng:
$$ M_s(F) = \sigma(f^{7 \times 7}([AvgPool(F); MaxPool(F)])) $$


In [ ]:
import os
import json
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
LR = 1e-3
IN_CHANNELS = 1
NUM_CLASSES = 10
MODEL_NAME = 'Attention CNN (CBAM)'
MODEL_KEY = 'attention'
DATASET_NAME = 'MNIST'


## Data Loading

In [ ]:
DATA_DIR = '../intel-sys-assignment-04/dataset/mnist_number/'
TRAIN_DIR = os.path.join(DATA_DIR, 'Augmented MNIST Training Set (400k)')
TEST_DIR = os.path.join(DATA_DIR, 'MNIST Validation Set (4k)')

train_transforms = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize(28),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_transforms = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize(28),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transforms)

torch.manual_seed(42)
indices = torch.randperm(len(full_train_dataset))[:20000]
train_dataset = Subset(full_train_dataset, indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = [str(i) for i in range(10)]
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Classes: {class_names}")


## Data Exploration

In [ ]:
# Data Exploration
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

class_found = {i: False for i in range(10)}
images_to_show = {}

for imgs, labels in train_loader:
    for img, label in zip(imgs, labels):
        lbl = label.item()
        if not class_found[lbl]:
            images_to_show[lbl] = img
            class_found[lbl] = True
        if all(class_found.values()):
            break
    if all(class_found.values()):
        break

for i in range(10):
    img = images_to_show[i].squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'Class: {i}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

from collections import Counter
train_labels = [full_train_dataset.targets[idx] for idx in indices.tolist()]
label_counts = Counter(train_labels)

plt.figure(figsize=(10, 5))
sns.barplot(x=list(label_counts.keys()), y=list(label_counts.values()))
plt.title('Class Distribution in Train Subset (20000)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()


## Model Architecture

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.shape
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg_out + max_out).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)
    def forward(self, x):
        return self.sa(self.ca(x))

class AttentionCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(in_channels, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.cbam1 = CBAM(32, reduction=4)
        self.pool1 = nn.MaxPool2d(2)
        self.block2 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.cbam2 = CBAM(64, reduction=8)
        self.pool2 = nn.MaxPool2d(2)
        self.block3 = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.cbam3 = CBAM(128, reduction=16)
        self.pool3 = nn.MaxPool2d(2)
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.pool1(self.cbam1(self.block1(x)))
        x = self.pool2(self.cbam2(self.block2(x)))
        x = self.pool3(self.cbam3(self.block3(x)))
        return self.classifier(x)

model = AttentionCNN(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")


## Training Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    return running_loss / total, correct / total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return running_loss / total, correct / total, all_preds, all_labels


## Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'time': 0}
start_time = time.time()

print(f"Starting training {MODEL_NAME} for {EPOCHS} epochs...")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} Acc: {test_acc:.4f}")

history['time'] = time.time() - start_time
print(f"Training completed in {history['time']:.2f} seconds.")


## Visualization

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['test_loss'], label='Test')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train')
plt.plot(history['test_acc'], label='Test')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()


## Final Evaluation

In [ ]:
_, _, y_pred, y_true = evaluate(model, test_loader, criterion)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()


## Save Results

In [ ]:
os.makedirs('results', exist_ok=True)
result_path = f"results/{DATASET_NAME.lower()}_{MODEL_KEY}.json"

results_dict = {
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "train_acc": history['train_acc'][-1],
    "test_acc": history['test_acc'][-1],
    "time": history['time']
}

with open(result_path, 'w') as f:
    json.dump(results_dict, f, indent=4)

print(f"Results saved to {result_path}")


## Conclusion\nTraining and evaluation completed successfully.